<a href="https://colab.research.google.com/github/VictoriaAzevedo/credit-score-analysis/blob/main/Projeto_Final_Credit_Score_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise do Perfil Financeiro dos Clientes e Fatores Associados ao Score de Crédito

### Projeto desenvolvido como parte do desafio final do Bootcamp de Analista de Dados da Re/Start. O objetivo deste estudo é analisar o perfil financeiro e comportamental dos clientes e identificar fatores associados à classificação do Score de Crédito.

## 1. Configuração do Ambiente

Configurações necessárias para execução do projeto, para acesso ao Google Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/bootcamp/projeto

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/bootcamp/projeto


## 2. Importação das Bibliotecas

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 3. Carregamento do arquivo

In [10]:
df = pd.read_csv("train.csv")

/tmp/ipykernel_646/3852424317.py:2: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("train.csv")


## 4. Análise Inicial da Base

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

In [34]:
resumo = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "Total de variáveis",
        "Variáveis categóricas (object)",
        "Variáveis inteiras (int64)",
        "Variáveis decimais (float64)",
        "Colunas com valores ausentes"
    ],
    "Valor": [
        df.shape[0],
        df.shape[1],
        (df.dtypes == "object").sum(),
        (df.dtypes == "int64").sum(),
        (df.dtypes == "float64").sum(),
        df.isnull().any().sum()
    ]
})

resumo

,Indicador,Valor
0,Total de registros,100000
1,Total de variáveis,28
2,Variáveis categóricas (object),20
3,Variáveis inteiras (int64),4
4,Variáveis decimais (float64),4
5,Colunas com valores ausentes,8


In [32]:
df.describe()

,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_Credit_Inquiries,Credit_Utilization_Ratio,Total_EMI_per_month
count,84998.000000,100000.000000,100000.00000,100000.000000,100000.000000,98035.000000,100000.000000,100000.000000
mean,4194.170850,17.091280,22.47443,72.466040,21.068780,27.754251,32.285173,1403.118217
std,3183.686167,117.404834,129.05741,466.422621,14.860104,193.177339,5.116875,8306.041270
min,303.645417,-1.000000,0.00000,1.000000,-5.000000,0.000000,20.000000,0.000000
25%,1625.568229,3.000000,4.00000,8.000000,10.000000,3.000000,28.052567,30.306660
50%,3093.745000,6.000000,5.00000,13.000000,18.000000,6.000000,32.305784,69.249473
75%,5957.448333,7.000000,7.00000,20.000000,28.000000,9.000000,36.496663,161.224249
max,15204.633333,1798.000000,1499.00000,5797.000000,67.000000,2597.000000,50.000000,82331.000000


In [33]:
df.describe(include='object')

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Num_of_Loan,Type_of_Loan,Num_of_Delayed_Payment,Changed_Credit_Limit,Credit_Mix,Outstanding_Debt,Credit_History_Age,Payment_of_Min_Amount,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
count,100000,100000,100000,90015,100000,100000,100000,100000,100000,88592,92998,100000,100000,100000,90970,100000,95521,100000,98800,100000
unique,100000,12500,8,10139,1788,12501,16,18940,434,6260,749,4384,4,13178,404,3,91049,7,98792,3
top,0x25fd5,CUS_0x942c,January,Stevex,38,#F%$D@*&8,_______,36585.12,3,Not Specified,19,_,Standard,1360.45,15 Years and 11 Months,Yes,__10000__,Low_spent_Small_value_payments,__-333333333333333333333333333__,Standard
freq,1,8,12500,44,2833,5572,7062,16,14386,1408,5327,2091,36479,24,446,52326,4305,25513,9,53174


Principais observações da estrutura dos dados

- O conjunto de dados possui **100.000 registros** distribuídos em **28 variáveis**, contendo informações cadastrais, financeiras e comportamentais dos clientes.

- Foram identificadas **oito variáveis com valores ausentes**, que serão tratadas durante a etapa de preparação dos dados (`Name`, `Monthly_Inhand_Salary`, `Type_of_Loan`, `Num_of_Delayed_Payment`, `Num_Credit_Inquiries`, `Credit_History_Age`, `Amount_invested_monthly` e `Monthly_Balance`).

- A base apresenta predominância de variáveis categóricas (`object`), enquanto apenas oito variáveis encontram-se em formato numérico (`int64` e `float64`).

- Algumas variáveis que representam valores numéricos encontram-se armazenadas como texto (`object`), indicando a necessidade de conversão para o tipo adequado antes da realização das análises.

- Foram identificados valores mínimos negativos em variáveis como `Num_Bank_Accounts` e `Delay_from_due_date`, que aparentam ser inconsistentes e serão investigados durante a etapa de limpeza dos dados.

- Algumas variáveis apresentam valores máximos bastante elevados, indicando possíveis valores extremos (outliers) ou inconsistências que deverão ser avaliadas durante a preparação dos dados.

- O conjunto de dados reúne informações cadastrais, financeiras e comportamentais, permitindo investigar diferentes fatores associados à classificação do score de crédito.

Conclusão da etapa

A análise inicial permitiu compreender a estrutura do conjunto de dados e identificar os principais desafios para as próximas etapas, especialmente relacionados ao tratamento de valores ausentes, conversão dos tipos de dados e investigação de possíveis inconsistências. A diversidade de informações disponíveis permitirá analisar a relação entre características financeiras e comportamentais dos clientes e sua classificação de score de crédito.

## 5. Pré-processamento dos Dados

In [36]:
## Criando uma cópia da base original para tratamento
df_limpo = df.copy()

### 5.1 Verificação de duplicidades

In [50]:
resumo_duplicidade = pd.DataFrame({
    "Indicador": [
        "Registros por ID",
        "Registros duplicados",
        "Duplicidade Cliente + Mês",
        "Meses analisados"
    ],
    "Valor": [
        df["Customer_ID"].nunique(),
        df.duplicated().sum(),
        df.duplicated(subset=["Customer_ID","Month"]).sum(),
        df["Month"].nunique()
    ]
})

resumo_duplicidade

,Indicador,Valor
0,Registros por ID,12500
1,Registros duplicados,0
2,Duplicidade Cliente + Mês,0
3,Meses analisados,8


In [51]:
df["Month"].unique()

array(['January', 'February', 'March', 'April', 'May', 'June', 'July',
       'August'], dtype=object)

Não foram identificados registros completamente duplicados, ou seja, nenhuma linha apresenta exatamente os mesmos valores em todas as variáveis. Também não foram encontradas duplicidades para a combinação **Customer_ID + Month**, indicando que cada cliente possui apenas um registro para cada mês analisado.

Foi identificado que a base contém aproximadamente **12.500 clientes distintos**, cada um com **8 registros** (100.000 ÷ 12.500 = 8). Essa característica confirma que a base possui uma estrutura temporal, permitindo acompanhar a evolução das informações financeiras dos clientes ao longo do tempo.

Esses resultados indicam boa consistência estrutural da base para as próximas etapas da análise.

### 5.2 Diagnóstico dos valores ausentes

In [37]:
faltantes = pd.DataFrame({'colunas':df.columns,
                      'tipo':df.dtypes,
                      'Qtde valores NaN':df.isna().sum(),
                      '% valores NaN':(df.isna().sum()/df.shape[0])*100,
                      'valores únicos por feature':df.nunique()})
faltantes = faltantes.reset_index()
faltantes

,index,colunas,tipo,Qtde valores NaN,% valores NaN,valores únicos por feature
0,ID,ID,object,0,0.000,100000
1,Customer_ID,Customer_ID,object,0,0.000,12500
2,Month,Month,object,0,0.000,8
3,Name,Name,object,9985,9.985,10139
4,Age,Age,object,0,0.000,1788
5,SSN,SSN,object,0,0.000,12501
6,Occupation,Occupation,object,0,0.000,16
7,Annual_Income,Annual_Income,object,0,0.000,18940
8,Monthly_Inhand_Salary,Monthly_Inhand_Salary,float64,15002,15.002,13235
9,Num_Bank_Accounts,Num_Bank_Accounts,int64,0,0.000,943


In [11]:
nulos = pd.DataFrame({
    "Quantidade": df.isnull().sum(),
    "Percentual (%)": (df.isnull().sum()/len(df)*100).round(2)
})

nulos = nulos[nulos["Quantidade"] > 0] \
            .sort_values("Percentual (%)", ascending=False)

nulos

,Quantidade,Percentual (%)
Monthly_Inhand_Salary,15002,15.00
Type_of_Loan,11408,11.41
Name,9985,9.98
Credit_History_Age,9030,9.03
Num_of_Delayed_Payment,7002,7.00
Amount_invested_monthly,4479,4.48
Num_Credit_Inquiries,1965,1.96
Monthly_Balance,1200,1.20


Plano de tratamento dos valores ausentes

Antes de iniciar o tratamento dos valores ausentes, foi realizada uma análise individual das variáveis que apresentam registros faltantes. O objetivo desta etapa é compreender a importância de cada variável para o projeto e definir uma estratégia de tratamento que minimize impactos na qualidade da análise e evite a introdução de vieses.

| Variável | % de Nulos | Importância | Hipótese inicial | Justificativa |
|-----------|-----------:|-------------|------------------|---------------|
| **Monthly_Inhand_Salary** | 15,00% | Alta | Investigar a relação com **Annual_Income** antes de definir a estratégia de tratamento. | Pode representar informação semelhante à renda anual, tornando desnecessária a manutenção de ambas as variáveis. |
| **Type_of_Loan** | 11,41% | Alta | Não realizar imputação inicialmente. Avaliar a manutenção dos valores ausentes ou a criação da categoria **"Não informado"**. | O preenchimento arbitrário pode alterar a distribuição dos tipos de empréstimo e introduzir viés na análise. |
| **Name** | 9,98% | Baixa | Avaliar a remoção da coluna. | O nome do cliente não contribui para explicar a classificação do Score de Crédito e não agrega valor às análises. |
| **Credit_History_Age** | 9,03% | Alta | Investigar o formato da variável antes de definir a estratégia de tratamento. | O tempo de histórico de crédito pode ser um fator relevante para explicar o Score de Crédito. |
| **Num_of_Delayed_Payment** | 7,00% | Alta | Não realizar imputação antes de analisar a distribuição dos dados. | Trata-se de uma variável comportamental importante e o preenchimento inadequado pode comprometer a análise. |
| **Amount_invested_monthly** | 4,48% | Média/Alta | Manter a variável e investigar sua relação com o Score de Crédito. | Pode fornecer informações relevantes sobre o perfil financeiro dos clientes e contribuir para a geração de insights. |
| **Num_Credit_Inquiries** | 1,96% | Alta | Manter a variável e avaliar posteriormente a melhor estratégia de tratamento. | O número de consultas ao histórico de crédito pode estar associado ao risco de crédito e influenciar a classificação do Score. |
| **Monthly_Balance** | 1,20% | Média | Avaliar posteriormente a melhor estratégia de tratamento. | O baixo percentual de valores ausentes permite diferentes abordagens, que serão definidas após investigação da variável. |

> **Observação:** As estratégias apresentadas nesta etapa representam hipóteses iniciais de tratamento. Nenhuma alteração será realizada na base de dados antes da investigação detalhada de cada variável e da avaliação dos possíveis impactos sobre as análises.

### 5.3 Diagnostico dos tipos de dados

In [17]:
## Pelo df.info() vimos que algumas colunas numéricas estão como object.
df_limpo.select_dtypes(include="object").columns

Index(['ID', 'Customer_ID', 'Month', 'Name', 'Age', 'SSN', 'Occupation',
       'Annual_Income', 'Num_of_Loan', 'Type_of_Loan',
       'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Credit_Mix',
       'Outstanding_Debt', 'Credit_History_Age', 'Payment_of_Min_Amount',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score'],
      dtype='object')

Algumas colunas não precisamos alterar, vamos focar apenas nessas:
'Age',
'Annual_Income',
'Monthly_Inhand_Salary',
'Num_of_Loan',
'Num_of_Delayed_Payment',
'Changed_Credit_Limit',
'Outstanding_Debt',
'Credit_History_Age',
'Amount_invested_monthly',
'Monthly_Balance'

In [39]:
df_limpo["Age"].unique()[:30]

array(['23', '-500', '28_', '28', '34', '54', '55', '21', '31', '33',
       '34_', '7580', '30', '30_', '24', '24_', '44', '45', '40', '41',
       '32', '33_', '35', '35_', '36', '39', '37', '181', '20', '46'],
      dtype=object)

In [20]:
df_limpo["Age"].str.contains("_", na=False).sum()

np.int64(4939)

Durante a investigação da coluna **Age**, foram identificados dois tipos de inconsistências:

- Presença de caracteres especiais (`_`) em **4.939 registros**, impedindo a conversão direta para valores numéricos.
- Existência de valores incompatíveis com o domínio da variável, como idades negativas e extremamente elevadas, indicando possíveis erros de registro que deverão ser tratados posteriormente.

In [41]:
df_limpo["Annual_Income"].unique()[:20]

array(['19114.12', '34847.84', '34847.84_', '143162.64', '30689.89',
       '30689.89_', '35547.71_', '35547.71', '73928.46', '131313.4',
       '10909427.0', '34081.38_', '34081.38', '114838.41', '114838.41_',
       '31370.8', '33751.27', '88640.24', '88640.24_', '54392.16'],
      dtype=object)

In [42]:
df_limpo["Annual_Income"].str.contains("_", na=False).sum()

np.int64(6980)

Análise da variável Annual_Income

A coluna **Annual_Income** encontra-se armazenada como texto (`object`), embora represente valores numéricos.

Foi identificada a presença do caractere especial (`_`) em **6.980 registros**, impedindo a conversão direta da variável para o tipo numérico (`float`).

Até o momento, não foram observados valores incompatíveis com o domínio da variável, sendo necessária apenas a remoção dos caracteres especiais antes da conversão.

In [28]:
df_limpo["Monthly_Inhand_Salary"].unique()[:30]

array([ 1824.84333333,            nan,  3037.98666667, 12187.22      ,
        2612.49083333,  2853.30916667,  5988.705     , 11242.78333333,
       10469.20775939,  2611.115     ,  9843.8675    ,  2825.23333333,
        2948.60583333,  7266.68666667,  4766.68      ,   519.12875   ,
        2415.855     ,  2942.14833333,  7591.59      ,  2898.385     ,
        8079.285     ,  7449.46934685,  1512.36166667,  1828.24      ,
        1074.58458333,  8873.4275    ,   782.03708333,  4720.92666667,
        1999.3075    ,  2697.51      ])

Foi identificado que diversas variáveis numéricas armazenadas como texto apresentam o caractere _, indicando um padrão de inconsistência na base de dados, dessa forma decidi avaliar todas as colunas de uma só vez

In [30]:
colunas_numericas = [
    "Age",
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Num_of_Loan",
    "Num_of_Delayed_Payment",
    "Changed_Credit_Limit",
    "Outstanding_Debt",
    "Amount_invested_monthly",
    "Monthly_Balance"
]

In [35]:
for coluna in colunas_numericas:
    caracteres = (
        df_limpo[coluna]
        .astype(str)
        .str.extractall(r'([^0-9.\-])')[0]
        .value_counts()
    )

    print(f"\n--- {coluna} ---")

    if caracteres.empty:
        print("Nenhum caractere inesperado encontrado.")
    else:
        print(caracteres)


--- Age ---
0
_    4939
Name: count, dtype: int64

--- Annual_Income ---
0
_    6980
Name: count, dtype: int64

--- Monthly_Inhand_Salary ---
0
n    30004
a    15002
Name: count, dtype: int64

--- Num_of_Loan ---
0
_    4785
Name: count, dtype: int64

--- Num_of_Delayed_Payment ---
0
n    14004
a     7002
_     2744
Name: count, dtype: int64

--- Changed_Credit_Limit ---
0
_    2091
Name: count, dtype: int64

--- Outstanding_Debt ---
0
_    1009
Name: count, dtype: int64

--- Amount_invested_monthly ---
0
_    17220
n     8958
a     4479
Name: count, dtype: int64

--- Monthly_Balance ---
0
n    2400
a    1200
_      36
Name: count, dtype: int64


A investigação das variáveis numéricas armazenadas como texto mostrou que o principal problema de formatação da base é a presença do caractere `_`, identificado em diversas colunas. Esse padrão de inconsistência impede a conversão direta dessas variáveis para tipos numéricos e será tratado na próxima etapa da preparação dos dados. A tabela abaixo resume os principais problemas encontrados e as ações previstas para cada variável.

| Variável | Tipo atual | Tipo esperado | Problema identificado | Ação prevista |
|-----------|------------|----------------|-----------------------|---------------|
| **Age** | `object` | `int` | Presença de caracteres especiais (`_`) e valores incompatíveis com a variável (idades negativas e muito elevadas). | Remover os caracteres especiais, converter para inteiro e validar os valores inconsistentes. |
| **Annual_Income** | `object` | `float` | Presença de caracteres especiais (`_`). | Remover os caracteres especiais e converter para `float`. |
| **Monthly_Inhand_Salary** | `float64` | `float` | Possui valores ausentes, embora o tipo de dado esteja correto. | Definir a estratégia de tratamento dos valores ausentes antes das análises. |
| **Num_of_Loan** | `object` | `int` | Presença de caracteres especiais (`_`). | Remover os caracteres especiais e converter para inteiro. |
| **Num_of_Delayed_Payment** | `object` | `int` | Presença de caracteres especiais (`_`) e valores ausentes. | Remover os caracteres especiais, converter para inteiro e definir a estratégia de tratamento dos valores ausentes. |
| **Changed_Credit_Limit** | `object` | `float` | Presença de caracteres especiais (`_`). | Remover os caracteres especiais e converter para `float`. |
| **Outstanding_Debt** | `object` | `float` | Presença de caracteres especiais (`_`). | Remover os caracteres especiais e converter para `float`. |
| **Credit_History_Age** | `object` | Numérico derivado | Armazenada em formato textual (ex.: `"22 Years and 3 Months"`). | Transformar a informação em uma variável numérica (meses ou anos) antes da análise. |
| **Amount_invested_monthly** | `object` | `float` | Presença de caracteres especiais (`_`) e valores ausentes. | Remover os caracteres especiais, converter para `float` e definir a estratégia de tratamento dos valores ausentes. |
| **Monthly_Balance** | `object` | `float` | Presença de poucos caracteres especiais (`_`) e valores ausentes. | Remover os caracteres especiais, converter para `float` e tratar os valores ausentes. |

### 5.4 Tratamento dos caracteres especiais

### 5.5 Conversão dos tipos

### 5.7 Validação das conversões

### 5.8 Tratamento dos valores ausentes

### 5.9 Análise de outliers

### 5.10 Base preparada para Análise Exploratória